# Full pipeline notebook

You can create a virtual environment to control the needed packages, but that should be optional. 

This notebooks takes the cutout tables from the other notebook, makes the actual cutouts and can compute sbf and colors.

## imports

In [ ]:
import numpy as np
import pandas as pd
import scipy as sc
import matplotlib.pyplot as plt
import sys

import os
import astropy.units as u
from astropy.io import fits
from photutils.isophote import EllipseGeometry
from IPython.display import clear_output

In [ ]:
# own function main pipeline
sys.path.append('sbf_pipeline/')
from euclid_sbf_pipeline import MainPipeline

sys.path.append('sbf_pipeline/functions/')
from fourierfunctions import appSBFmagnitude
from colorextraction import resize, trim, computeColorCombinedMask

In [ ]:
filt = 'VIS'
# filt = 'H'
table_name = 'your_table_name'
table = pd.read_csv(f'/media/user/path_to_your_cutout_table/{table_name}_{filt}_matched.csv')
table

In [ ]:
# Check you effective radius range
print(table['re'].min(),table['re'].max())

In [ ]:
# If needed: lets set a certain size limit

# re_lim_min = 3
# table = table[table['re']>=re_lim_min]

# re_lim_max = 15
# table = table[table['re']<=re_lim_max]


table

In [ ]:
# checking with observation ids are in the sample
oids = []
for idx, (index, sub_df) in enumerate(table.groupby(['observation_id'])):
    oids.append(str(index[0]))
oids = np.array(oids)
print(oids)

## Make the cutouts

In [ ]:
import cutout_utils_v2 as ut

In [ ]:
# this function is for the corresponding PSFs
def split_tiles(img):
    if img.shape == (189, 189):

        tiles = (
            img.reshape(9, 21, 9, 21)   # (rows, tile_h, cols, tile_w)
               .transpose(0, 2, 1, 3)   # (rows, cols, tile_h, tile_w)
               .reshape(-1, 21, 21)     # (81, 21, 21)
        )
        
        sums = tiles.sum(axis=(1, 2), keepdims=True)

        # Avoid division by zero for empty tiles
        sums[sums == 0] = 1

        # Normalize each tile independently
        tiles /= sums

        return tiles
    else:
        print(f"Wrong psf input shape: {img.shape} instead of 189,189")

In [ ]:
# cutouts folder must be defined still
cutout_folder = f"/media/user/input_data/{table_name}/{filt}/"

In [ ]:
product_type = 'stack'
cutout_size = 're'


if filt == 'VIS': 
    def cutout_func(x):
    # for VIS data + field x*90+75
    # for NISP data + field change to x*30+25
        # return x * 180 + 100   # habas sample
        return x * 144 + 100   # fds sample
elif filt == 'H':
    def cutout_func(x):
        #have to test these numbers still
        # return x * 60 + 100   # habas sample
        return x * 48 + 100   # fds sample

optional_cutout_params = {}
resize_params=None

ra_colname, dec_colname = 'right_ascension', 'declination'
radec_colnames =  {"ra_colname": ra_colname, "dec_colname": dec_colname}

In [ ]:
# make the cutouts
for idx, (index, sub_df) in enumerate(table.groupby(['observation_id'])):
    oid = str(index[0])
    
    # incase of specific tile selection uncommented the next line
    # if oid == '6119':
    output_location = cutout_folder + oid + '/'
    os.makedirs(output_location, exist_ok=True)

    # getting the image from volume
    folder = sub_df["datalabs_path"].iloc[0] + '/'
    hdul = fits.open(folder + sub_df["file_name"].iloc[0])

    #check if doing constant radius cutouts or variable and apply cutout function to each row in small df
    if type(cutout_size) == str:
        ap_res = sub_df.apply(lambda x: ut.cutoutsGroup(x[radec_colnames["ra_colname"]], x[radec_colnames["dec_colname"]], x["file_name"], hdul, product_type, x[cutout_size], cutout_func, optional_cutout_params, output_location, resize_params), axis=1)
    if filt == 'H':
        if type(cutout_size) == str:
            ap_res = sub_df.apply(lambda x: ut.cutoutsGroup(x[radec_colnames["ra_colname"]], x[radec_colnames["dec_colname"]], x["file_name"], hdul, "weight", x[cutout_size], cutout_func, optional_cutout_params, output_location, resize_params), axis=1)

    hdul.close()

    if filt == 'VIS':
        for file in os.listdir(folder):
            if 'WGT' in file:
                hdul = fits.open(folder + file)

                #check if doing constant radius cutouts or variable and apply cutout function to each row in small df
                if type(cutout_size) == str:
                    ap_res = sub_df.apply(lambda x: ut.cutoutsGroup(x[radec_colnames["ra_colname"]], x[radec_colnames["dec_colname"]], file, hdul, "weight", x[cutout_size], cutout_func, optional_cutout_params, output_location, resize_params), axis=1)

                hdul.close()

    if filt == 'VIS':
        for file in os.listdir(folder):
            if 'PSF' in file:
                psfoption = fits.getdata(folder + file)
                # psfoptiondat = psfoption[0].data
                tiles = split_tiles(psfoption)
                print("PSF tiles shape:", np.shape(tiles))
                np.save(output_location + 'psf.npy', tiles)
    if filt == 'H':
        tiles = []
        for file in os.listdir(folder):
            if ('PSF' in file) and ('CAL' in file) and ('I_H' in file):
                hdu = fits.open(folder + file)
                for idx in range(len(hdu)-1):
                    tiles.append(hdu[idx+1].data.astype(np.float64))
                hdu.close()
        tiles = np.stack(tiles)
        print("PSF tiles shape:", np.shape(tiles))
        np.save(output_location + 'psf.npy', tiles)


## Compute SBF

In [ ]:
# i recommend not doing all possible observation ids (tiles) in one go, so choose one here:
oid = '6119'
print(oid)

In [ ]:
input_folder = f'/media/user/input_data/{table_name}/{filt}/{oid}'  #this should be your cutouts folder 
output_folder = f'/media/user/output_data/{table_name}/{oid}'

psfs = np.load(input_folder +'/psf.npy')
if filt == "H":
    psfs_res = []
    for psf in psfs:
        psf /= np.sum(psf)    
        psfs_res.append(resize(trim(psf,6, do_print=False), s=6, plot=False))
    psfs = np.stack(psfs_res, axis=0)
print(np.shape(psfs))

gals = sorted(os.listdir(input_folder))
ras = table['right_ascension'][table['observation_id'] == float(oid)]

In [ ]:
# plots the image outputs in the notebook
plot_plots = True

In [ ]:
sbf_list = []
for gal_ra in ras:
    for g in gals:
        if (str(gal_ra) in g) and ('STK' in g):
            gal = g
        if (str(gal_ra) in g) and ('WGT' in g):
            wgt = g

    print(gal)
    print(wgt)


    start = gal.find("CUTOUT_")
    if start != -1:
        gal_name = gal[start + len("CUTOUT_"):-5]
        print(gal_name)
    del start
    
    # define all relevant paths

    data_path = '{}/{}'.format(input_folder,gal)  #filter is already defined above
    mask_path = '{}/{}'.format(input_folder,wgt)  #filter is already defined above
    file_path = '{}/{}/{}/return_files'.format(output_folder,gal_name,filt)
    image_path = '{}/{}/{}/images'.format(output_folder,gal_name,filt)
    fits_path = '{}/{}/{}/fits_files'.format(output_folder,gal_name,filt)

    paths = [file_path, image_path, fits_path]
    for path in paths:
        try:
            os.makedirs(path)
        except FileExistsError:
            pass
    del paths



    # setting last settings
    geometry = None
    if filt == 'H':
        try:
            file_path_VIS = '{}/{}/{}/return_files'.format(output_folder,gal_name,'VIS')
            geom_params = np.loadtxt(file_path_VIS + '/geometry')
            geometry =  EllipseGeometry(*geom_params)
            geometry.x0 = int(np.round(geometry.x0/3))
            geometry.y0 = int(np.round(geometry.y0/3))
            geometry.sma = int(np.round(geometry.sma/3))
        except:
            print("No geometry file found")
            geometry = None
        
    inv_image_sourcemasking = True
    cat = None
    SN_psfitting = 3
    
    maxarea_sourcemask = None
    
    # run the pipeline on the galaxy
    sbf, std_p0, sbfmag, noise, sbf_cor, sbfcor_err, sbfmag_cor, sma, rout, total_bckgr, mzp, geometry, Npix = MainPipeline(
             data_path, mask_path=mask_path, file_path=file_path, field_path=None, field_from_data=True,image_path=image_path, 
             make_plots=True, plot_plots=plot_plots, psf=psfs, geometry=geometry,
             maxarea_sourcemask=maxarea_sourcemask, filter=filt, 
             background_estimation=True, background_estimation_median=False, background=None, 
             ellipsefitter='v7', sma_rescale=0.8,  
             maskbackgroundsourcesweighted=True, SN_psfitting=SN_psfitting, 
             manual_nri_mask=cat, galmolcorr=True , galmolsmooth=False,  
             residual_sources_estimation=True, fits_path=fits_path, inv_image_sourcemasking=inv_image_sourcemasking)
    print("sbf, std_p0, sbfmag, noise, sbf_cor, sbfcor_err, sbfmag_cor, sma, rout, total_bckgr, mzp, geometry, Npix")
    print(sbf, std_p0, sbfmag, noise, sbf_cor, sbfcor_err, sbfmag_cor, sma, rout, total_bckgr, mzp, geometry, Npix)

    sbfmagerr = 2.5/(sbf*np.log(10)) * std_p0
    try:
        sbfmagcorerr = 2.5/(sbf_cor*np.log(10)) * (std_p0 + sbfcor_err)
    except:
        sbfmagcorerr = 0
    print(f"\nUncorrected \n{sbfmag}+-{sbfmagerr}")
    print(f"\nCorrected \n{sbfmag_cor}+-{sbfmagcorerr}")
    sbf_list.append([sbfmag, sbfmagerr, sbfmag_cor, sbfmagcorerr])
    
    # Incase you want to check the plots of every galaxy before continuing use the following line:
    
    input("Press Enter to continue...")

    clear_output(wait=True)

## Compute color

### H band just background subtracted images

Incase you did not want to compute SBF in the H band, then use this to just create some data for computing the colors

In [ ]:
filt = 'H'

In [ ]:
input_folder = f'/media/user/input_data/{table_name}/{filt}/{oid}' #cutouts folder
output_folder = f'/media/user/output_data/{table_name}/{oid}'

gals = sorted(os.listdir(input_folder))
ras = table['right_ascension'][table['observation_id'] == float(oid)]

In [ ]:
for gal_ra in ras:
    for g in gals:
        if (str(gal_ra) in g) and ('STK' in g):
            gal = g
        if (str(gal_ra) in g) and ('WGT' in g):
            wgt = g

    print(gal)
    print(wgt)


    start = gal.find("CUTOUT_")
    if start != -1:
        gal_name = gal[start + len("CUTOUT_"):-5]
        print(gal_name)
    del start
    
    # define all relevant paths

    data_path = '{}/{}'.format(input_folder,gal)  #filter is already defined above
    mask_path = '{}/{}'.format(input_folder,wgt)  #filter is already defined above
    file_path = '{}/{}/{}/return_files'.format(output_folder,gal_name,filt)
    image_path = '{}/{}/{}/images'.format(output_folder,gal_name,filt)

    paths = [file_path, image_path]
    for path in paths:
        try:
            os.makedirs(path)
        except FileExistsError:
            pass
    del paths

        

    
    # run the pipeline on the galaxy
    _,_,_,_,_ = MainPipeline(
             data_path, mask_path=mask_path, file_path=file_path, field_path=None, field_from_data=True,image_path=image_path, 
             make_plots=True, plot_plots=True, psf=None, geometry=None, premstop=True,
             maxarea_sourcemask=None, filter=filt, 
             background_estimation=True, background_estimation_median=False, background=None, 
             ellipsefitter='v7', sma_rescale=0.8,  
             maskbackgroundsourcesweighted=True, SN_psfitting=None, 
             manual_nri_mask=None, galmolcorr=True , galmolsmooth=False,  
             residual_sources_estimation=True, fits_path=None, inv_image_sourcemasking=False)
    
    input("Press Enter to continue...")

    clear_output(wait=True)

### Computing color

In [ ]:
for gal_ra in ras:
    for g in gals:
        if (str(gal_ra) in g) and ('STK' in g):
            gal = g

    print(gal)

    start = gal.find("CUTOUT_")
    if start != -1:
        gal_name = gal[start + len("CUTOUT_"):-5]
        print(gal_name)
    del start

    pathVIS = '{}/{}/VIS/return_files'.format(output_folder,gal_name)
    pathH = '{}/{}/H/return_files'.format(output_folder,gal_name)
    maskVIS = np.loadtxt(pathVIS + '/combined_final_mask')
    try:
        maskH = np.loadtxt(pathH + '/combined_final_mask')
    except:
        maskH = np.loadtxt(pathH + '/mask_initial')
    dataVIS = np.loadtxt(pathVIS + '/data_background_subtracted')
    dataH = np.loadtxt(pathH + '/data_background_subtracted')

    mzpVIS = np.loadtxt(pathVIS + '/background')[2]
    mzpH =  np.loadtxt(pathH + '/background')[2]
    print('mzp:',mzpVIS,mzpH)

    color,dataBmasked,dataRmasked = computeColorCombinedMask(dataVIS, dataH, maskVIS, maskH, mzpVIS, mzpH, do_print=True)
    np.savetxt(f"{output_folder}/{gal_name}/color", [color])
    
    input("Press Enter to continue...")

    clear_output(wait=True)

In [ ]:
hdr = fits.getheader('input_data/test_files/FCC188/FCC188.fits')
hdr

In [ ]:
datast = fits.getdata('input_data/habas_fornax/VIS/6118/VIS_SWL-STK-000-000000-0000000_-CUTOUT_54.268988_-35.590119.fits')
datast -= np.nanmedian(datast)
source_maskst = ~np.isfinite(datast)

In [ ]:
data = fits.getdata('input_data/test_files/FCC188/FCC188.fits')
data -= np.nanmedian(data)
source_mask = ~np.isfinite(data)

In [ ]:
import sep
bkg = sep.Background(data.astype(np.float64), mask=source_mask, bw=64, bh=64, fw=3, fh=3)
print(bkg.globalrms)
bkg = sep.Background(datast.astype(np.float64), mask=source_maskst, bw=64, bh=64, fw=3, fh=3)
print(bkg.globalrms)

In [ ]:
print(np.nanstd(data.flatten()))

In [ ]:
plt.imshow(data, norm='log', origin='lower')
plt.colorbar()
plt.show()

In [ ]:
residual_final, model_final, geometry, isolist = MainFitEllipseModel(data, mask_cr=source_mask, geometry=None, mgecenter='med', method='v7',
                                                                            make_plots=True, plot_plots=True, final=True, image_path=None)

In [ ]:
from astropy.coordinates import SkyCoord

In [ ]:
coor = SkyCoord("14 58 48.7186892424","+02 01 24.710276208", unit=(u.hourangle, u.degree))
coor


### Testing cutout issue

In [ ]:
def chooseBestFile(df_res_dup, radec_colnames):
    """Choosing best file for sources with multiple by checking where the source is closest to image centre"""
    
    min_dist_per_plotcode = df_res_dup.groupby([radec_colnames["ra_colname"], 'filter_name'], as_index=False, sort=False)['dist'].idxmin()
    df_final = df_res_dup.loc[min_dist_per_plotcode["dist"]]
    
    # check here for these border cases
    if check_border_cases:
        for i in range(len(df_final)):
            if df_final['dist'][i] > dist_center_border:  # select those 
            # recompute the distance, by taking the second best frame as well, and not using radial distance, but distance from edge 
            cornerA, cornerB, cornerC, cornerD = df_final['fov'][i]
            distA = sqrt of (df_final[radec_colnames["ra_colname"]] - cornerA[0])**2 + (df_final[radec_colnames["dec_colname"]] - cornerA[1])**2   # repeat for all corners
            
            # pick the two closest corners
            
            # add the distances from the source to the corners and subbtract the distance between those two border corners - call value border_subtr
            
            # find the one to smallest corresponding filter name in df_res_dup and compute the same
            
            # if the border_subtr is larger for the one to smallest filter name, than replace it in the df_final and print a comment that that happened
    
    return df_final

In [ ]:
# filt = 'VIS'
filt = 'H'

In [ ]:
# import coordinate table
table_name = 'habas_fornax' #fds_xii_2022_matched
table = pd.read_csv(f'/media/user/input_data/tables/{table_name}_{filt}_matched.csv')

In [ ]:
table.keys()

In [ ]:
table

In [ ]:
import cutout_utils_v2 as ut

In [ ]:
def split_tiles(img):
    if img.shape == (189, 189):

        tiles = (
            img.reshape(9, 21, 9, 21)   # (rows, tile_h, cols, tile_w)
               .transpose(0, 2, 1, 3)   # (rows, cols, tile_h, tile_w)
               .reshape(-1, 21, 21)     # (81, 21, 21)
        )
        
        sums = tiles.sum(axis=(1, 2), keepdims=True)

        # Avoid division by zero for empty tiles
        sums[sums == 0] = 1

        # Normalize each tile independently
        tiles /= sums

        return tiles
    else:
        print(f"Wrong psf input shape: {img.shape} instead of 189,189")

In [ ]:
# output_folder must be defined still
output_folder = f"/media/user/input_data/{table_name}/{filt}/"

In [ ]:
arraytest = np.arange(8)
arraytest[2:-2]

In [ ]:
product_type = 'stack'
cutout_size = 're'


if filt == 'VIS': 
    def cutout_func(x):
    # for VIS data + field x*90+75
    # for NISP data + field change to x*30+25
        return x * 180 + 100
elif filt == 'H':
    def cutout_func(x):
        #have to test these numbers still
        return x * 60 + 100

optional_cutout_params = {}
resize_params=None

ra_colname, dec_colname = 'right_ascension', 'declination'
radec_colnames =  {"ra_colname": ra_colname, "dec_colname": dec_colname}

In [ ]:
# make the cutouts
for idx, (index, sub_df) in enumerate(table.groupby(['observation_id'])):
    oid = str(index[0])
    
    # incase of specific tile selection uncommented the next line
    # if oid == '6118':
    output_location = output_folder + oid + '/'
    os.makedirs(output_location, exist_ok=True)

    # getting the image from volume
    folder = sub_df["datalabs_path"].iloc[0] + '/'
    hdul = fits.open(folder + sub_df["file_name"].iloc[0])

    #check if doing constant radius cutouts or variable and apply cutout function to each row in small df
    if type(cutout_size) == str:
        ap_res = sub_df.apply(lambda x: ut.cutoutsGroup(x[radec_colnames["ra_colname"]], x[radec_colnames["dec_colname"]], x["file_name"], hdul, product_type, x[cutout_size], cutout_func, optional_cutout_params, output_location, resize_params), axis=1)
    if filt == 'H':
        if type(cutout_size) == str:
            ap_res = sub_df.apply(lambda x: ut.cutoutsGroup(x[radec_colnames["ra_colname"]], x[radec_colnames["dec_colname"]], x["file_name"], hdul, "weight", x[cutout_size], cutout_func, optional_cutout_params, output_location, resize_params), axis=1)

    hdul.close()

    if filt == 'VIS':
        for file in os.listdir(folder):
            if 'WGT' in file:
                hdul = fits.open(folder + file)

                #check if doing constant radius cutouts or variable and apply cutout function to each row in small df
                if type(cutout_size) == str:
                    ap_res = sub_df.apply(lambda x: ut.cutoutsGroup(x[radec_colnames["ra_colname"]], x[radec_colnames["dec_colname"]], file, hdul, "weight", x[cutout_size], cutout_func, optional_cutout_params, output_location, resize_params), axis=1)

                hdul.close()

    if filt == 'VIS':
        for file in os.listdir(folder):
            if 'PSF' in file:
                psfoption = fits.getdata(folder + file)
                # psfoptiondat = psfoption[0].data
                tiles = split_tiles(psfoption)
                print("PSF tiles shape:", np.shape(tiles))
                np.save(output_location + 'psf.npy', tiles)
    if filt == 'H':
        tiles = []
        for file in os.listdir(folder):
            if ('PSF' in file) and ('CAL' in file) and ('I_H' in file):
                hdu = fits.open(folder + file)
                for idx in range(len(hdu)-1):
                    tiles.append(hdu[idx+1].data.astype(np.float64))
                hdu.close()
        tiles = np.stack(tiles)
        print("PSF tiles shape:", np.shape(tiles))
        np.save(output_location + 'psf.npy', tiles)


In [ ]:
folder = '/data/euclid_idr1/DR1/R1/NIR/6118/'

In [ ]:
# psfoptiondat = psfoption[0].data
# psfoptionhdr = psfoption[0].header

In [ ]:
# psfoptionhdr

In [ ]:
foldertest = '/data/euclid_idr1/DR1/R1/NIR/6118/'
# folder = '/data/euclid_idr1/DR1/R1/VIS/6105/'  #05
for file in os.listdir(foldertest):
    if 'PSF' in file:
        if 'CAL' in file:
            psfoption = fits.getheader(foldertest + file)
        # psfoptiondat = psfoption[0].data
        # tiles = split_tiles(psfoption)
        # print("PSF tiles shape:", np.shape(tiles))
        # np.save(output_location + 'psf.npy', tiles)

In [ ]:
psfoption

In [ ]:
import os
from astropy.io import fits

In [ ]:
sorted(os.listdir(foldertest))

In [ ]:
for file in os.listdir(folder):
    if ("STK" in file) and ('IMAGE_H' in file):
    # if ("PSF" in file) and ("CAL" in file) and ('I_H' in file):
        hdu = fits.open(folder+file)
data = hdu[1].data#.astype(np.float64)
header = hdu[1].header

In [ ]:
len(hdu)

In [ ]:
hdu.info()

In [ ]:
header

In [ ]:
data

In [ ]:
plt.imshow(data)

In [ ]:
hdu.close()

In [ ]:
from astropy.coordinates import SkyCoord
from astropy.nddata.utils import Cutout2D
import astropy.units as u
from astropy import wcs
ra, dec = 54.268988, -35.590119
# ra,dec = 53.7983, -35.32306

In [ ]:
print(SkyCoord(ra, dec, frame='icrs', unit="deg").ra.to(u.hourangle), SkyCoord(ra, dec, frame='icrs', unit="deg").dec)

In [ ]:
cutout = Cutout2D(data= data, position = SkyCoord(ra, dec, frame='icrs', unit="deg"), size=1000, wcs=wcs.WCS(header))

In [ ]:
test = np.array((79.30172424204214,-26.95226392445689,80.14298154560207,-26.886687602257673,80.05755517559157,-26.040533392920445,79.2224235052122,-26.105628775678845)).reshape(4,2)
print(test)
fig, ax = plt.subplots(figsize=(10,10))
ax.scatter(79.68184043689,-26.49555927513, c='red')
ax.scatter(test[:,0],test[:,1])
ax.plot(test[:,0],test[:,1])
ax.xaxis.set_inverted(True) 
plt.show()

In [ ]:
plt.hist(cutout_wgt.data.flatten(), bins=100)
plt.show()
plt.close()

In [ ]:
plt.imshow(cutout_wgt.data, origin='lower')
plt.colorbar()
plt.show()
plt.imshow(cutout.data, norm='log', origin='lower')
plt.colorbar()
plt.show()
plt.close('all')

In [ ]:
plt.imshow(cutout_wgt.data, origin='lower')
plt.colorbar()
plt.show()
plt.imshow(cutout.data, norm='log', origin='lower')
plt.colorbar()
plt.show()
plt.close('all')

In [ ]:
plt.imshow(cutout_wgt.data, origin='lower')
plt.colorbar()
plt.show()
plt.imshow(cutout.data, norm='log', origin='lower')
plt.colorbar()
plt.show()
plt.close('all')

In [ ]:
fig= plt.figure()
plt.imshow(psfoptiondat[:21, :21], norm='log')
plt.colorbar()
plt.show()
plt.close()

fig= plt.figure()
plt.imshow(psfoptiondat, norm='log')
plt.colorbar()
plt.show()
plt.close()

In [ ]:
tiles = split_tiles(psfoptiondat)
np.save(output_location + 'psf.npy', tiles)
print(tiles.shape)



# optional small check:
# for i in [0,1,79,80]:
#     print("sum:", np.sum(tiles[i]))
#     fig= plt.figure()
#     plt.imshow(tiles[i], norm='log')
#     plt.colorbar()
#     plt.title(i)
#     plt.show()
#     plt.close()

In [ ]:
res = [cuotutSubdf(sub_df, product_type, cutout_size, cutout_func, optional_cutout_params, output_location, resize_params, radec_colnames) for index, sub_df in tqdm_notebook(df.groupby(['file_name']), desc = "Number of files processed")]

In [ ]:
ut.makeCutoutsByTile()